In [1]:
import pandas as pd
import math
import itertools
import tqdm
import operator

from PATHS import DATA_FOLDER
from data_prep import merge_open_close
from info_extraction import calc_profit, calc_profit_with_condition

In [2]:
open_df = pd.read_csv(DATA_FOLDER + 'M15/2024-2026/smaller_spike/trailing_with_ADX-ATR/ATR_FILTER_GRATERTAN_VALUE/open_trades_test.csv',sep=';')
close_df = pd.read_csv(DATA_FOLDER + 'M15/2024-2026/smaller_spike/trailing_with_ADX-ATR/ATR_FILTER_GRATERTAN_VALUE/closed_trades_test.csv',sep=';')

In [3]:
merge_df = merge_open_close(open_df,close_df)

In [6]:
merge_df.columns

Index(['Test_ID', 'Type', 'Ticket', 'Symbol_open', 'OpenPrice', 'OpenTime',
       'fast_MA', 'fast_MA_prev', 'fast_EMA_diff', 'body', 'range',
       'BodyRatio', 'MA', 'slow_MA', 'MACD', 'ADX', 'SAR', 'ICHI', 'ENVELOPES',
       'RSI_7', 'RSI_14', 'RSI_21', 'RSI_28', 'RSI_35', 'STOCH', 'MOMENTUM',
       'CCI', 'WPR', 'RVI', 'Bands', 'ATR', 'STD', 'VOLUME', 'OBV', 'AD',
       'MFI', 'AO', 'AC', 'DeMarker', 'Alligator', 'Symbol_close',
       'ClosePrice', 'Profit', 'CloseTime'],
      dtype='str')

In [4]:
merge_df.sort_values(by='Profit')

,Test_ID,Type,Ticket,Symbol_open,OpenPrice,OpenTime,fast_MA,fast_MA_prev,fast_EMA_diff,body,...,AD,MFI,AO,AC,DeMarker,Alligator,Symbol_close,ClosePrice,Profit,CloseTime
32111,0.015_1.0_0.030_4.0,Long,168,ETHUSD,4808.2,2025.08.24 22:23:41,1.009706,1.015144,-0.005438,72.5,...,288.587678,41.747920,36.763235,-53.929706,0.369911,1.002194,ETHUSD,4676.0,-147.342006,2025.08.25 02:42:32
23999,0.015_1.0_0.030_3.0,Long,168,ETHUSD,4808.2,2025.08.24 22:23:41,1.009706,1.015144,-0.005438,72.5,...,288.587678,41.747920,36.763235,-53.929706,0.369911,1.002194,ETHUSD,4676.0,-147.342006,2025.08.25 02:42:32
15887,0.015_1.0_0.030_2.0,Long,168,ETHUSD,4808.2,2025.08.24 22:23:41,1.009706,1.015144,-0.005438,72.5,...,288.587678,41.747920,36.763235,-53.929706,0.369911,1.002194,ETHUSD,4676.0,-147.342006,2025.08.25 02:42:32
7775,0.015_1.0_0.030_1.0,Long,168,ETHUSD,4808.2,2025.08.24 22:23:41,1.009706,1.015144,-0.005438,72.5,...,288.587678,41.747920,36.763235,-53.929706,0.369911,1.002194,ETHUSD,4676.0,-147.342006,2025.08.25 02:42:32
31828,0.010_1.0_0.030_4.0,Long,274,ETHUSD,4802.5,2025.08.24 21:57:25,1.019653,1.021976,-0.002322,86.8,...,289.124705,60.161286,101.155588,-15.612176,0.439455,1.000187,ETHUSD,4676.4,-131.562534,2025.08.25 02:42:31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23841,0.015_1.0_0.030_3.0,Long,10,ETHUSD,3384.2,2024.03.05 22:43:13,1.041135,1.047728,-0.006593,31.8,...,105.344348,37.542464,-231.176471,-64.443824,0.053478,1.095268,ETHUSD,3796.8,407.753500,2024.03.06 14:46:02
15345,0.010_1.0_0.030_2.0,Long,15,ETHUSD,3367.5,2024.03.05 22:42:48,1.046298,1.052924,-0.006626,31.8,...,105.866769,37.542464,-231.176471,-64.443824,0.053478,1.100700,ETHUSD,3796.8,411.727495,2024.03.06 14:46:02
7233,0.010_1.0_0.030_1.0,Long,15,ETHUSD,3367.5,2024.03.05 22:42:48,1.046298,1.052924,-0.006626,31.8,...,105.866769,37.542464,-231.176471,-64.443824,0.053478,1.100700,ETHUSD,3796.8,411.727495,2024.03.06 14:46:02
31569,0.010_1.0_0.030_4.0,Long,15,ETHUSD,3367.5,2024.03.05 22:42:48,1.046298,1.052924,-0.006626,31.8,...,105.866769,37.542464,-231.176471,-64.443824,0.053478,1.100700,ETHUSD,3796.8,411.727495,2024.03.06 14:46:02


In [4]:
filter_candidates = {
    "ADX": [
        (operator.lt, [18, 20]),
        (operator.gt, [25, 30]),
    ],

    "RSI_7": [
        (operator.gt, [80, 85]),
        (operator.lt, [20, 15]),
    ],

    "RSI_14": [
        (operator.gt, [70, 75]),
        (operator.lt, [30, 25]),
    ],

    "STOCH": [
        (operator.gt, [85, 90]),
        (operator.lt, [15, 10]),
    ],

    "CCI": [
        (operator.gt, [100, 150, 200]),
        (operator.lt, [-100, -150, -200]),
    ],

    "MFI": [
        (operator.gt, [80, 85]),
        (operator.lt, [20, 15]),
    ],

    "WPR": [
        (operator.gt, [-20, -10]),
        (operator.lt, [-80, -90]),
    ],

    "DeMarker": [
        (operator.gt, [0.7, 0.8]),
        (operator.lt, [0.3, 0.2]),
    ],

    # These should ideally be normalized first:
    # ATR_pct = ATR / Close
    "ATR": [
        (operator.gt, [0.03, 0.05]),
        (operator.lt, [0.008, 0.005]),
    ],
}

In [ ]:
df = merge_df.copy()

# ---------------------------
# 1. Base win/loss split
# ---------------------------
base = (
    df.groupby(["Test_ID", "Type"], as_index=False)
    .agg(
        Total_Trades=("Profit", "count"),
        Win=("Profit", lambda x: x[x > 0].sum()),
        Loss=("Profit", lambda x: x[x <= 0].sum()),
    )
)

base["Profit"] = base["Win"] + base["Loss"]
out_df = base.copy()


# ------------------------------------------------------------
# PRECOMPUTE ALL SINGLE MASKS
# ------------------------------------------------------------

condition_cache = []

for indicator in filter_candidates:

    for instance in filter_candidates[indicator]:

        operation, values = instance

        OPERATOR = (
            'LT'
            if f'{operation}' == '<built-in function lt>'
            else 'GT'
        )

        for value in values:

            mask = operation(df[indicator], value)

            condition_cache.append({
                "mask": mask,
                "name": f"{indicator}_{OPERATOR}_{value}"
            })


# ------------------------------------------------------------
# STORE ALL RESULTS (NO MERGE IN LOOP)
# ------------------------------------------------------------

results = []

COMBINATION_SIZES = [2, 3, 4]


for r in COMBINATION_SIZES:

    total_combinations = math.comb(len(condition_cache), r)

    for cond_group in tqdm.tqdm(
        itertools.combinations(condition_cache, r),
        total=total_combinations,
        desc=f"{r}-condition combos"
    ):

        # ----------------------------------------------------
        # BUILD FINAL MASK (ALL AND)
        # ----------------------------------------------------

        final_mask = cond_group[0]["mask"].copy()

        for cond in cond_group[1:]:
            final_mask &= cond["mask"]

        df_filter = df[final_mask]

        if df_filter.empty:
            continue

        # ----------------------------------------------------
        # NAME
        # ----------------------------------------------------

        combo_name = "__AND__".join(
            cond["name"] for cond in cond_group
        )

        # ----------------------------------------------------
        # GROUPBY
        # ----------------------------------------------------

        Filter = (
            df_filter.groupby(["Test_ID", "Type"], as_index=False)
            .agg(
                Win=("Profit", lambda x: x[x > 0].sum()),
                Loss=("Profit", lambda x: x[x <= 0].sum()),
            )
        )

        Filter = Filter.rename(columns={
            "Win": f"{combo_name}_Win",
            "Loss": f"{combo_name}_Loss"
        })

        results.append(Filter)


# ------------------------------------------------------------
# FINAL MERGE (ONLY ONCE)
# ------------------------------------------------------------

for r in results:
    out_df = out_df.merge(
        r,
        on=["Test_ID", "Type"],
        how="left"
    )

out_df = out_df.fillna(0)


# ------------------------------------------------------------
# COMPUTE METRICS AFTER MERGE
# ------------------------------------------------------------

win_cols = [c for c in out_df.columns if c.endswith("_Win")]

for col in win_cols:

    base_name = col.replace("_Win", "")
    loss_col = base_name + "_Loss"

    out_df[base_name + "_Profit"] = (
        out_df[col] + out_df[loss_col]
    )

    out_df[base_name + "_Win_Difference"] = (
        out_df[col] - out_df["Win"]
    )

    out_df[base_name + "_Loss_difference"] = (
        out_df[loss_col] - out_df["Loss"]
    )

    out_df[base_name + "_Improvement"] = (
        out_df[base_name + "_Profit"] - out_df["Profit"]
    )

    out_df[base_name + "_Loss_Win_difference"] = (
        abs(out_df[base_name + "_Loss_difference"])
        / abs(out_df[col].replace(0, 1e-9))
    )

4-condition combos: 100%|██████████| 73815/73815 [38:24<00:00, 32.03it/s]  
C:\Users\User\AppData\Local\Temp\ipykernel_23876\1834798216.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[base_name + "_Win_Difference"] = (
C:\Users\User\AppData\Local\Temp\ipykernel_23876\1834798216.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out_df[base_name + "_Loss_difference"] = (
C:\Users\User\AppData\Local\Temp\ipykernel_23876\1834798216.py:145: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

In [92]:
out_df

,Test_ID,Type,Total_Trades,Win,Loss,Profit,ADX_LT_18__AND__ADX_LT_20_Win,ADX_LT_18__AND__ADX_LT_20_Loss,ADX_LT_18__AND__RSI_7_GT_80_Win,ADX_LT_18__AND__RSI_7_GT_80_Loss,...,DeMarker_GT_0.7__AND__DeMarker_GT_0.8__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Profit,DeMarker_GT_0.7__AND__DeMarker_GT_0.8__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Win_Difference,DeMarker_GT_0.7__AND__DeMarker_GT_0.8__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Loss_difference,DeMarker_GT_0.7__AND__DeMarker_GT_0.8__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Improvement,DeMarker_GT_0.7__AND__DeMarker_GT_0.8__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Loss_Win_difference,DeMarker_LT_0.3__AND__DeMarker_LT_0.2__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Profit,DeMarker_LT_0.3__AND__DeMarker_LT_0.2__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Win_Difference,DeMarker_LT_0.3__AND__DeMarker_LT_0.2__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Loss_difference,DeMarker_LT_0.3__AND__DeMarker_LT_0.2__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Improvement,DeMarker_LT_0.3__AND__DeMarker_LT_0.2__AND__ATR_LT_0.008__AND__ATR_LT_0.005_Loss_Win_difference
0,0.01_1.00_0.01_0.05_0.01_0,Short,357,7266.576458,-10548.333750,-3281.757292,536.148958,-763.381042,0.0,-45.677083,...,-277.843375,-7266.576458,10270.490375,3003.913917,1.027049e+13,137.016250,-6658.185542,10076.959083,3418.773542,1.656330e+01
1,0.01_1.00_0.01_0.05_0.01_1,Long,1640,34333.119000,-43202.284917,-8869.165917,2186.436667,-3087.860125,0.0,0.000000,...,208.487917,-33294.949000,42372.602833,9077.653833,4.081471e+01,-443.920583,-33449.203208,41874.448542,8425.245333,4.737380e+01
2,0.01_1.00_0.01_0.05_0.02_0,Short,357,7266.576458,-10548.333750,-3281.757292,536.148958,-763.381042,0.0,-45.677083,...,-277.843375,-7266.576458,10270.490375,3003.913917,1.027049e+13,137.016250,-6658.185542,10076.959083,3418.773542,1.656330e+01
3,0.01_1.00_0.01_0.05_0.02_1,Long,1640,34333.119000,-43202.284917,-8869.165917,2186.436667,-3087.860125,0.0,0.000000,...,208.487917,-33294.949000,42372.602833,9077.653833,4.081471e+01,-443.920583,-33449.203208,41874.448542,8425.245333,4.737380e+01
4,0.01_1.00_0.01_0.05_0.03_0,Short,357,7266.576458,-10548.333750,-3281.757292,536.148958,-763.381042,0.0,-45.677083,...,-277.843375,-7266.576458,10270.490375,3003.913917,1.027049e+13,137.016250,-6658.185542,10076.959083,3418.773542,1.656330e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0.05_1.00_0.05_0.05_0.03_1,Long,22,2186.599500,-770.933500,1415.666000,0.000000,0.000000,0.0,0.000000,...,0.000000,-2186.599500,770.933500,-1415.666000,7.709335e+11,0.000000,-2186.599500,770.933500,-1415.666000,7.709335e+11
246,0.05_1.00_0.05_0.05_0.04_0,Short,24,907.640000,-2280.426833,-1372.786833,0.000000,0.000000,0.0,0.000000,...,0.000000,-907.640000,2280.426833,1372.786833,2.280427e+12,0.000000,-907.640000,2280.426833,1372.786833,2.280427e+12
247,0.05_1.00_0.05_0.05_0.04_1,Long,22,2186.599500,-770.933500,1415.666000,0.000000,0.000000,0.0,0.000000,...,0.000000,-2186.599500,770.933500,-1415.666000,7.709335e+11,0.000000,-2186.599500,770.933500,-1415.666000,7.709335e+11
248,0.05_1.00_0.05_0.05_0.05_0,Short,24,907.640000,-2280.426833,-1372.786833,0.000000,0.000000,0.0,0.000000,...,0.000000,-907.640000,2280.426833,1372.786833,2.280427e+12,0.000000,-907.640000,2280.426833,1372.786833,2.280427e+12


In [86]:
top = -1
idx = 0
top_col = None
for col in out_df.columns:
    if '_Profit' in col:
        profit = out_df[col].iloc[out_df[col].idxmax()]
        if top < profit:
            top = profit
            idx = out_df[col].idxmax()
            top_col = col
print(top, idx,top_col)

3873.4131249999996 71 ADX_GT_25__AND__ATR_LT_0.008_Profit
